# Decision Engine

This notebook combines predictions from the traffic and energy models into a rule-based decision layer.

In [1]:
import joblib
import pandas as pd

def get_traffic_prediction(input_row):
    model = joblib.load('../models/traffic_forecasting_model.pkl')
    features = joblib.load('../models/traffic_features.pkl')
    row = pd.DataFrame([input_row])[features]
    pred = float(model.predict(row)[0])
    return {'predicted_volume': round(pred), 'current_volume': input_row.get('traffic_lag_1', 0)}

def get_energy_prediction(input_row):
    model = joblib.load('../models/energy_forecasting_model.pkl')
    features = joblib.load('../models/energy_features.pkl')
    row = pd.DataFrame([input_row])[features]
    pred = float(model.predict(row)[0])
    return {'predicted_energy_wh': round(pred, 1)}

def multi_objective_decision_engine(traffic, energy, environment, weather):
    actions = []
    reasons = []
    pred_vol = traffic.get('predicted_volume', 0)
    road_cap = traffic.get('road_capacity', 6000)
    congestion_ratio = pred_vol / max(road_cap, 1)

    if congestion_ratio >= 0.85:
        actions.append('ALERT: Activate alternate route guidance')
        reasons.append(f'Traffic at {congestion_ratio * 100:.0f}% of road capacity')
    elif congestion_ratio >= 0.65:
        actions.append('Extend green signal phase on main corridor')
        reasons.append('Traffic approaching moderate congestion threshold')

    if energy.get('energy_anomaly', False):
        actions.append('Investigate abnormal energy usage')
        reasons.append('Anomaly detector flagged unusual consumption')

    if weather.get('rain_mm', 0) > 5:
        actions.append('Activate wet-road speed advisory signs')
        reasons.append('Heavy rain may reduce road friction')

    if not actions:
        actions = ['Maintain current operations']
        reasons = ['All systems within normal operating range']

    return {
        'recommended_actions': actions,
        'reasons': reasons,
        'urgency': 'HIGH' if any('ALERT' in a for a in actions) else 'NORMAL'
    }

print('Decision engine ready.')

Decision engine ready.


In [2]:
sample_traffic = {
    'hour': 8, 'day_of_week': 1, 'month': 6, 'weekend': 0,
    'holiday_enc': 0, 'temp': 25, 'rain_1h': 2.0, 'snow_1h': 0.0, 'clouds_all': 20,
    'weather_main_enc': 0, 'weather_desc_enc': 0,
    'traffic_lag_1': 3500, 'traffic_lag_2': 3400, 'traffic_lag_3': 3300,
    'traffic_lag_6': 3200, 'traffic_lag_12': 3000, 'traffic_lag_24': 2800,
    'rolling_mean_3': 3300, 'rolling_mean_6': 3200, 'rolling_mean_12': 3100, 'rolling_std_6': 100
}

traffic_result = get_traffic_prediction(sample_traffic)
traffic_result['road_capacity'] = 6000
energy_result = {'predicted_energy_wh': 750, 'energy_anomaly': False}
decision = multi_objective_decision_engine(traffic_result, energy_result, {}, {'rain_mm': 8})
print(decision)

{'recommended_actions': ['Activate wet-road speed advisory signs'], 'reasons': ['Heavy rain may reduce road friction'], 'urgency': 'NORMAL'}
